##### Define notebook Parameters

Pass through th pl_worker

In [72]:
# Framework-compatible parameters with manual fallbacks
import json
import uuid

# Basic notebook runtime values
task_id = 50
task_name = "Load RMDD to silver"
lineage_id = str(uuid.uuid4())
limit_rows_for_debugging = False
prev_wm = "1900-01-01 00:00:00"
curr_wm = "2026-06-30 00:00:00"
error_results = []

# Source settings > what data to read
source_settings = json.dumps(
    {
        "source_name": "CAN_RMDD",
        "source_type": "sql_database",
        "source_location": "CAN_RMDD-236c5edd-b846-437b-b99b-39e8e56a0a5e",
        "source_schema": "dbo",
        "table_config": [
            {
                "source_table": "Address",
                "target_table": "test_silver_rmdd_address",
                "target_pk": ["address_id"],
                "is_incremental": 1,
                "incremental_column": "LastUpdatedDateGMT",
                "transformation_notebook": "nb_transform_silver_rmdd_address"
            },
            {
                "source_table": "Client",
                "target_table": "test_silver_rmdd_client",
                "target_pk": ["client_number", "member_firm_code"],
                "is_incremental": 1,
                "incremental_column": "LastUpdatedDateGMT",
                "transformation_notebook": "nb_transform_silver_rmdd_client"
            },
            {
                "source_table": "PersonRelationship",
                "target_table": "test_silver_rmdd_person_relationship",
                "target_pk": ["person_number_1", "member_firm_code_1", "person_number_2", "member_firm_code_2"],
                "is_incremental": 1,
                "incremental_column": "LastUpdatedDateGMT",
                "transformation_notebook": "nb_transform_silver_rmdd_person_relationship"
            },
            {
                "source_table": "Person",
                "target_table": "test_silver_rmdd_person",
                "target_pk": ["person_number", "member_firm_code", "person_key"],
                "is_incremental": 1,
                "incremental_column": "LastUpdatedDateGMT",
                "transformation_notebook": "nb_transform_silver_rmdd_person"
            }
        ]
    }
)


# Target settings > here/how to write
target_settings = json.dumps(
    {
        "target_lakehouse": "lh_canada_global_mds",
        "target_schema": "silver_rmdd",
        "target_location": "Files/test/rmdd",
        "target_load_strategy": "merge-scd1"
    }
)

# Connection settings > how to connect/authenticate
source_connection_settings = json.dumps(
    {
        "server_name": "hbvkj6b5fvzenlsxgtupezx6wq-4qzy5g26bakuhffkkiyl64azc4.database.fabric.microsoft.com"
    }
)

StatementMeta(, 147ea718-aa72-4e63-9547-36bdd6495198, 130, Finished, Available, Finished, False)

##### Define functions and logging

Used for import functions

In [73]:
%run nb_transformation_shared_functions

StatementMeta(, 147ea718-aa72-4e63-9547-36bdd6495198, 138, Finished, Available, Finished, True)

In [74]:
# Set up standard framework logging
setup_logging()
logger = logging.getLogger("LoadRMDDToSilver")
logger.setLevel(logging.INFO)

StatementMeta(, 147ea718-aa72-4e63-9547-36bdd6495198, 139, Finished, Available, Finished, False)

##### Define variables or parameters

Can manually run or be through worker

In [75]:
# Accept either JSON strings from pipeline or already-parsed dicts for manual runs
source_settings = json.loads(source_settings) if isinstance(source_settings, str) else source_settings
target_settings = json.loads(target_settings) if isinstance(target_settings, str) else target_settings
source_connection_settings = json.loads(source_connection_settings) if isinstance(source_connection_settings, str) else source_connection_settings

# Source configuration
source_name = source_settings.get("source_name")
source_type = source_settings.get("source_type")
source_location = source_settings.get("source_location")
source_schema = source_settings.get("source_schema")
table_config = source_settings.get("table_config", [])

# Target configuration
target_lakehouse = target_settings.get("target_lakehouse")
target_schema = target_settings.get("target_schema")
target_location = target_settings.get("target_location")
target_load_strategy = target_settings.get("target_load_strategy")

# Source connection configuration
server_name = (
    source_connection_settings.get("server_name")
    or source_connection_settings.get("jdbc_hostname")
    or source_connection_settings.get("server")
)

# Debug output
if limit_rows_for_debugging:
    print(f"source_name: {source_name}")
    print(f"source_type: {source_type}")
    print(f"source_location: {source_location}")
    print(f"source_schema: {source_schema}")
    print(f"target_schema: {target_schema}")
    print(f"target_location: {target_location}")
    print(f"target_load_strategy: {target_load_strategy}")
    display(table_config)

StatementMeta(, 147ea718-aa72-4e63-9547-36bdd6495198, 140, Finished, Available, Finished, False)

##### Build source connection

Creates connection to get source data

In [76]:
# Build source connection if this source needs one
jdbc_url, connection_properties = build_source_connection(
    source_type=source_type,
    source_location=source_location,
    source_connection_settings=source_connection_settings
)

# Debug output
if limit_rows_for_debugging:
    print(f"jdbc_url: {jdbc_url}")

StatementMeta(, 147ea718-aa72-4e63-9547-36bdd6495198, 141, Finished, Available, Finished, False)

##### Ingest source table

Read and dedup clean source

In [77]:
# Read source tables and create src_* temp views
source_dfs = {}

for cfg in table_config:
    # Get the source table name from the table config
    source_table = cfg.get("source_table")

    # Create a standard view name and read the source table
    source_view = f"src_{to_snake_case(source_table)}"
    source_full_name = f"{source_schema}.{source_table}"
    
    # Read data from sql/shortcut/lakehouse
    df_source = read_source_table(
        source_full_name=source_full_name,
        jdbc_url=jdbc_url,
        connection_properties=connection_properties
    )

    # Clean, register, and save the source table for later steps
    df_source = remove_exact_duplicates(df_source)
    df_source.createOrReplaceTempView(source_view)
    source_dfs[source_table] = df_source

    # Debug output
    if limit_rows_for_debugging:
        print(f"Source preview for {source_view} ({source_full_name})")
        display(df_source.limit(3))

StatementMeta(, 147ea718-aa72-4e63-9547-36bdd6495198, 142, Finished, Available, Finished, False)

##### Transform source table

Applies mapping and transformation as needed

In [78]:
# Transform source tables into target shape
mapped_dfs = {}

for cfg in table_config:
    # Get the source and target table names from config
    source_table = cfg.get("source_table")
    target_table = cfg.get("target_table")
    transformation_notebook = cfg.get("transformation_notebook")

    # Build standard source and mapped view names
    target_entity = to_snake_case(source_table)
    source_view = f"src_{target_entity}"
    map_view = f"map_{target_entity}"

    # Build the watermark filter for incremental loads
    watermark_column = cfg.get("incremental_column")
    is_incremental = cfg.get("is_incremental", 0)
    watermark_filter = "1 = 1"

    if is_incremental == 1 and watermark_column:
        watermark_filter = f"s.{watermark_column} > '{prev_wm}' AND s.{watermark_column} <= '{curr_wm}'"

    try:
        # Run table-specific transformation when provided
        if transformation_notebook:

            notebook_timeout_seconds = 600

            mssparkutils.notebook.run(
                transformation_notebook,
                notebook_timeout_seconds,
                {
                    "source_view": source_view,
                    "map_view": map_view,
                    "watermark_filter": watermark_filter
                }
            )

            mapped_dfs[target_table] = spark.table(map_view)

        # If no transformation is provided, use the ingested source data as-is
        else:
            mapped_dfs[target_table] = source_dfs[source_table]

        # Debug output
        if limit_rows_for_debugging:
            print(f"Mapped preview for {target_table}")
            display(mapped_dfs[target_table].limit(3))

    except Exception as error:
        #  Keep the notebook running and skip only the failed table
        add_run_issue(
            run_issue_results=error_results,
            task_id=task_id,
            task_name=task_name,
            lineage_id=lineage_id,
            source_table=source_table,
            target_table=target_table,
            stage="transformation",
            severity="ERROR",
            message=error
        )

        continue

if error_results:
    print("Tables skipped because of errors:")
    display(error_results)

StatementMeta(, 147ea718-aa72-4e63-9547-36bdd6495198, 143, Finished, Available, Finished, False)

Skipping test_silver_rmdd_person_relationship. Transformation failed.


Tables skipped because of errors:


SynapseWidget(Synapse.DataFrame, e027b27b-e360-41c5-b6b5-14b834294a06)

##### Add metadata to source table

Applies metadata and _hashed_pk

In [79]:
# Add metadata to final target data
target_dfs = {}

for cfg in table_config:
    # Get source and target details
    source_table = cfg.get("source_table")
    target_table = cfg.get("target_table")
    target_pk = cfg.get("target_pk", [])

    # Use transformed data when it exists; otherwise use the ingested source data
    df_target = mapped_dfs.get(target_table, source_dfs[source_table])

    # Add hashes only for merge loads
    if target_load_strategy == "merge-scd1":
        df_target = add_hash_columns(df_target, target_pk)

    # Add standard framework metadata columns
    df_target = add_metadata_v2(
        df=df_target,
        source_system=source_name,
        source_table=f"{source_schema}.{source_table}",
        target_table=target_table,
        lineage_id=lineage_id
    )

    # Save final DataFrame for validation and load
    target_dfs[target_table] = df_target

    # Debug output
    if limit_rows_for_debugging:
        print(f"Target preview for {target_table}")
        display(df_target.limit(3))

StatementMeta(, 147ea718-aa72-4e63-9547-36bdd6495198, 144, Finished, Available, Finished, False)

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `person_number_1` cannot be resolved. Did you mean one of the following? [`PersonID1`, `PersonID2`, `Active`, `Description`, `LastUpdatedBy`].;
'Project [PersonRelationshipID#23081, RelationshipTypeID#23082, PersonID1#23083, PersonID2#23084, Description#23085, PrimaryFlag#23090, Active#23087, LastUpdatedDateGMT#23088, LastUpdatedBy#23089, md5(concat_ws(|, coalesce(cast('person_number_1 as string), <NULL>), coalesce(cast('member_firm_code_1 as string), <NULL>), coalesce(cast('person_number_2 as string), <NULL>), coalesce(cast('member_firm_code_2 as string), <NULL>))) AS _hashed_pk#24435]
+- Deduplicate [Description#23085, LastUpdatedBy#23089, PersonID1#23083, PersonID2#23084, RelationshipTypeID#23082, LastUpdatedDateGMT#23088, PrimaryFlag#23090, Active#23087, PersonRelationshipID#23081]
   +- Project [PersonRelationshipID#23081, RelationshipTypeID#23082, PersonID1#23083, PersonID2#23084, Desc

##### Check duplicates

Return duplicates before load_to_target 

In [ ]:
# Check duplicate business keys before load
for cfg in table_config:
    # Get table definition from source_settings
    target_table = cfg.get("target_table")
    target_pk = cfg.get("target_pk")
    
    # Call nb_transformation_shared_functions
    validate_duplicates(target_dfs[target_table], target_pk)

StatementMeta(, 147ea718-aa72-4e63-9547-36bdd6495198, -1, Cancelled, , Cancelled, True)

##### Load to target

Runs load to target based on load_strategy

In [ ]:
# Load final DataFrames to silver taxonomy tables
load_results = []

for cfg in table_config:
    # Get table definition from source_settings
    target_table = cfg.get("target_table")

    # Set target table and location
    target_full_name = f"{target_schema}.{target_table}"
    target_location_path = f"{target_location}/{target_table}"
    
    # Lookup target table on target_dfs
    df_target = target_dfs[target_table]

    # Call nb_transformation_shared_functions
    result = load_to_target(df_target, target_full_name, target_load_strategy, target_location_path)

    # Save metrics for pipeline
    load_results.append({
        "target_table": target_full_name,
        "rows_inserted": result["rows_inserted"],
        "rows_updated": result["rows_updated"],
        "rows_read": result["rows_read"],
        "rows_copied": result["rows_copied"]
    })

display(load_results)

StatementMeta(, 147ea718-aa72-4e63-9547-36bdd6495198, -1, Cancelled, , Cancelled, True)